<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture - Agents, MCP, and AI Governance on Databricks
## Overview
This lecture establishes the foundations for building AI agents on Databricks. We cover what agents are, how they use tools registered in Unity Catalog, how the Model Context Protocol (MCP) enables tool discovery and execution, and how AI Gateway provides governance across the stack.

## Learning Objectives
By the end of this lecture, you will be able to:
1. Define what AI agents are and identify their core components
2. Explain how Unity Catalog functions serve as governed agent tools
3. Describe how MCP enables agents to discover and call tools at runtime
4. Identify common tool patterns: structured retrieval, unstructured retrieval, code interpreter, and external connections
5. Understand the role of AI Gateway for governance and access control

## A. What are AI Agents?

### A1. Single Agents

An **AI agent** is an intelligent software system that can perceive its environment, make decisions, and take actions to achieve specific goals. Unlike traditional AI systems that require continuous inputs from users, AI agents are autonomous systems that can:

- **Reason** about complex problems and situations
- **Plan** sequences of actions to achieve objectives
- **Adapt** their behavior based on new information
- **Interact** with external systems and data sources
- **Learn** from experience to improve future performance

A very simple example of agent framework is shown below. Here, a system prompt and the user's prompt are passed to the agent, where it then uses tools to help plan and process information to return back to the user.

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 680 385" role="img" style="font-family: sans-serif;">
  <title>Agent Framework</title>
  <desc>Agent Framework containing Generative AI models, Tools, and Sequence of decisions, taking System Prompt and User Prompt as input and producing Response and Output.</desc>
  <defs>
    <marker id="af-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <rect x="25" y="15" width="176" height="74" rx="8" fill="#ffffff" stroke="#1B3139" stroke-width="1.5"/>
  <text x="113" y="40" text-anchor="middle" dominant-baseline="central" font-size="14" font-weight="600" fill="#0b2026">System Prompt</text>
  <text x="113" y="64" text-anchor="middle" dominant-baseline="central" font-size="14" font-weight="600" fill="#0b2026">+ User Prompt</text>
  <rect x="479" y="15" width="176" height="74" rx="8" fill="#ffffff" stroke="#1B3139" stroke-width="1.5"/>
  <text x="567" y="52" text-anchor="middle" dominant-baseline="central" font-size="14" font-weight="600" fill="#0b2026">Response/Output</text>
  <path d="M113 89 L113 107" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#af-arrow)"/>
  <path d="M567 107 L567 89" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#af-arrow)"/>
  <rect x="15" y="110" width="650" height="265" rx="14" fill="#F9F7F4" stroke="#1B3139" stroke-width="1.5" stroke-opacity="0.4"/>
  <text x="34" y="137" text-anchor="start" font-size="14" font-weight="700" fill="#FF5F46">Agent Framework</text>
  <rect x="32" y="154" width="278" height="181" rx="10" fill="#ffffff"/>
  <rect x="48" y="188" width="504" height="70" rx="12" fill="#2272B4" fill-opacity="0.10"/>
  <text x="171" y="176" text-anchor="middle" dominant-baseline="central" font-size="14" font-weight="600" fill="#0b2026">Generative AI</text>
  <foreignObject x="86" y="194" width="58" height="58">
    <div xmlns="http://www.w3.org/1999/xhtml" style="width:100%;height:100%;">
      <img src="./Includes/images/agent-icon.png" style="width:100%;height:100%;display:block;" alt="AI brain icon"/>
    </div>
  </foreignObject>
  <foreignObject x="210" y="194" width="58" height="58">
    <div xmlns="http://www.w3.org/1999/xhtml" style="width:100%;height:100%;">
      <img src="./Includes/images/agent-icon.png" style="width:100%;height:100%;display:block;" alt="AI brain icon"/>
    </div>
  </foreignObject>
  <text x="387" y="223" text-anchor="middle" dominant-baseline="central" font-size="50" font-weight="500" font-style="italic" fill="#0b2026">fx</text>
  <rect x="438" y="202" width="54" height="42" rx="3" fill="#ffffff" stroke="#0b2026" stroke-width="1.5"/>
  <line x1="438" y1="213" x2="492" y2="213" stroke="#0b2026" stroke-width="1"/>
  <circle cx="444" cy="208" r="1.2" fill="#0b2026"/>
  <circle cx="449" cy="208" r="1.2" fill="#0b2026"/>
  <circle cx="454" cy="208" r="1.2" fill="#0b2026"/>
  <circle cx="465" cy="229" r="10" fill="none" stroke="#0b2026" stroke-width="1.3"/>
  <ellipse cx="465" cy="229" rx="10" ry="3.5" fill="none" stroke="#0b2026" stroke-width="1"/>
  <line x1="465" y1="219" x2="465" y2="239" stroke="#0b2026" stroke-width="1"/>
  <rect x="48" y="188" width="504" height="70" rx="12" fill="none" stroke="#2272B4" stroke-width="1.5" stroke-dasharray="6 3"/>
  <rect x="32" y="154" width="278" height="181" rx="10" fill="none" stroke="#618794" stroke-width="1.2"/>
  <text x="542" y="204" text-anchor="end" dominant-baseline="central" font-size="14" font-weight="600" fill="#0b2026">Tools</text>
  <text x="115" y="273" text-anchor="middle" dominant-baseline="central" font-size="14" font-weight="500" fill="#0b2026">Image</text>
  <text x="115" y="289" text-anchor="middle" dominant-baseline="central" font-size="14" font-weight="500" fill="#0b2026">Generation</text>
  <text x="239" y="273" text-anchor="middle" dominant-baseline="central" font-size="14" font-weight="500" fill="#0b2026">Language</text>
  <text x="239" y="289" text-anchor="middle" dominant-baseline="central" font-size="14" font-weight="500" fill="#0b2026">Model</text>
  <text x="171" y="318" text-anchor="middle" dominant-baseline="central" font-size="14" fill="#0b2026">(GenAI Examples)</text>
  <path d="M438 262 L438 274" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#af-arrow)"/>
  <path d="M452 274 L452 262" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#af-arrow)"/>
  <rect x="334" y="278" width="310" height="90" rx="14" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/>
  <rect x="354" y="292" width="272" height="12" rx="6" fill="#618794"/>
  <rect x="354" y="310" width="272" height="12" rx="6" fill="#618794"/>
  <rect x="354" y="328" width="272" height="12" rx="6" fill="#00A972"/>
  <text x="489" y="356" text-anchor="middle" dominant-baseline="central" font-size="14" fill="#0b2026">Sequence of decisions</text>
</svg>
</div>

           
### A2. Multi-Agent Supervisors

An example of a common multi-agent architecture is the **supervisor-worker** (also called router-delegate) pattern:

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 680 310" role="img" style="font-family: sans-serif;">
  <title>Supervisor-Worker Multi-Agent Architecture</title>
  <desc>A Supervisor Agent receives user requests and delegates to specialized worker agents: Data Agent, Research Agent, and General Agent.</desc>
  <defs>
    <marker id="sw-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <rect x="15" y="15" width="650" height="280" rx="14" fill="#F9F7F4" stroke="#1B3139" stroke-width="1.5" stroke-opacity="0.4"/>
  <text x="34" y="42" text-anchor="start" font-size="14" font-weight="700" fill="#FF5F46">Multi-Agent System</text>
  <rect x="240" y="60" width="200" height="60" rx="10" fill="#ffffff" stroke="#618794" stroke-width="1.2"/>
  <text x="340" y="82" text-anchor="middle" font-size="14" font-weight="600" fill="#0b2026">Supervisor Agent</text>
  <text x="340" y="104" text-anchor="middle" font-size="11" fill="#618794">Routes and synthesizes</text>
  <path d="M270 120 L152 165" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#sw-arrow)"/>
  <path d="M340 120 L340 165" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#sw-arrow)"/>
  <path d="M410 120 L528 165" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#sw-arrow)"/>
  <text x="275" y="143" text-anchor="middle" font-size="12" font-style="italic" fill="#618794">Delegates to</text>
  <rect x="48" y="168" width="584" height="110" rx="12" fill="#2272B4" fill-opacity="0.10"/>
  <rect x="48" y="168" width="584" height="110" rx="12" fill="none" stroke="#2272B4" stroke-width="1.5" stroke-dasharray="6 3"/>
  <text x="622" y="183" text-anchor="end" font-size="14" font-weight="600" fill="#0b2026">Workers</text>
  <rect x="68" y="194" width="160" height="66" rx="8" fill="#ffffff" stroke="#1B5162" stroke-width="1.2"/>
  <text x="148" y="220" text-anchor="middle" font-size="13" font-weight="600" fill="#0b2026">Data Agent</text>
  <text x="148" y="242" text-anchor="middle" font-size="11" fill="#618794">UC functions, SQL</text>
  <rect x="260" y="194" width="160" height="66" rx="8" fill="#ffffff" stroke="#1B5162" stroke-width="1.2"/>
  <text x="340" y="220" text-anchor="middle" font-size="13" font-weight="600" fill="#0b2026">Research Agent</text>
  <text x="340" y="242" text-anchor="middle" font-size="11" fill="#618794">AI Search, docs</text>
  <rect x="452" y="194" width="160" height="66" rx="8" fill="#ffffff" stroke="#1B5162" stroke-width="1.2"/>
  <text x="532" y="220" text-anchor="middle" font-size="13" font-weight="600" fill="#0b2026">General Agent</text>
  <text x="532" y="242" text-anchor="middle" font-size="11" fill="#618794">Knowledge, reasoning</text>
</svg>
</div>

**How it works:**
1. The **supervisor** receives the user's request and analyzes intent
2. It **delegates** to the appropriate worker agent (or handles simple requests itself)
3. The worker executes with its specialized tools and returns a result
4. The supervisor **synthesizes** the response and returns it to the user

For example, in the OpenAI Agents SDK, this delegation is implemented using **handoffs**:

```python
from agents import Agent

supervisor = Agent(
    name="Supervisor",
    instructions="Route requests to the appropriate specialist.",
    handoffs=[data_agent, business_agent],
)
```

### A3. Single-Agent vs. Multi-Agent

Factor                 | Single Agent                                              | Multi-Agent
---------------------- | --------------------------------------------------------- | -----------------------------------------------
Number of tools        | Works best when total tools is modest (≈ up to 8–10); beyond that, tool selection and prompt management start to degrade (heuristic, not a hard limit). | Better when you have many tools spanning distinct capabilities (often >10) and need clear separation between them.
Domain complexity      | One focused or tightly related domain.                   | Multiple distinct domains or verticals.
Instruction length     | Shorter, coherent system prompt that a single agent can reliably follow. | Would require a very long, fragmented, or conflicting prompt if kept in one agent.
Failure isolation      | Failures are acceptable as one unit; easier to reason about as a whole. | Need to isolate, debug, and replace individual agent capabilities without changing the whole system.
Team ownership         | Single team owns most or all behaviors.                  | Different teams own different domains, tools, or behaviors and need independent evolution.
Latency                | Minimal overhead; typically fewer model calls and simpler orchestration. | Extra routing/orchestration calls add latency and token cost.
Scalability / parallelism | Scales fine while domain and tools remain bounded; limited parallelism within one agent’s context window. | Better for parallelizing work across domains, distributing context, and scaling capabilities, at the cost of more coordination complexity.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
Start with a single agent. Only decompose into multiple agents when you observe tool selection errors, instruction conflicts, or when different teams need to own different agent capabilities independently.
</p>
</div>
</div>
</div>


## B. Tools and Common Patterns

           
           
### B1. Model Context Protocol (MCP)

On Databricks, [Model Context Protocol (MCP)](https://modelcontextprotocol.io/docs/getting-started/intro) is an open standard for connecting AI agents to tools and context such as data, functions, and external services. Databricks managed MCP servers ([AWS](https://docs.databricks.com/aws/en/generative-ai/mcp/managed-mcp/) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/mcp/managed-mcp) | [GCP](https://docs.databricks.com/gcp/en/generative-ai/mcp/managed-mcp)) are Databricks-hosted, ready-to-use MCP endpoints for AI Search Search, Genie spaces, Databricks SQL, and Unity Catalog functions. They let agents connect to these tools through a standard interface while Unity Catalog permissions remain enforced and usage can be monitored and managed through Unity AI Gateway.


<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 680 375" role="img" style="font-family: sans-serif;">
  <title>MCP on Databricks</title>
  <desc>User query flows into an agent on Databricks that acts as an MCP client. The agent discovers and calls tools from managed MCP servers (Genie for structured data, AI Search for unstructured data, UC Functions for custom logic) and returns a response to the user. All traffic governed by Unity Catalog and AI Gateway.</desc>
  <defs>
    <marker id="mcp-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>

  <rect x="25" y="15" width="176" height="74" rx="8" fill="#ffffff" stroke="#1B3139" stroke-width="1.5"/>
  <text x="113" y="52" text-anchor="middle" dominant-baseline="central" font-size="14" font-weight="600" fill="#0b2026">User Query</text>

  <rect x="479" y="15" width="176" height="74" rx="8" fill="#ffffff" stroke="#1B3139" stroke-width="1.5"/>
  <text x="567" y="52" text-anchor="middle" dominant-baseline="central" font-size="14" font-weight="600" fill="#0b2026">Response</text>

  <path d="M113 89 L113 107" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#mcp-arrow)"/>
  <path d="M567 107 L567 89" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#mcp-arrow)"/>

  <rect x="15" y="110" width="650" height="250" rx="14" fill="#F9F7F4" stroke="#1B3139" stroke-width="1.5" stroke-opacity="0.4"/>
  <text x="34" y="137" text-anchor="start" font-size="14" font-weight="700" fill="#FF5F46">Databricks</text>

  <rect x="32" y="154" width="210" height="160" rx="10" fill="#ffffff" stroke="#618794" stroke-width="1.2"/>
  <text x="137" y="176" text-anchor="middle" dominant-baseline="central" font-size="14" font-weight="600" fill="#0b2026">Agent</text>
  <foreignObject x="107" y="196" width="60" height="60">
    <div xmlns="http://www.w3.org/1999/xhtml" style="width:100%;height:100%;">
      <img src="./Includes/images/agent-icon.png" style="width:100%;height:100%;display:block;" alt="AI agent icon"/>
    </div>
  </foreignObject>
  <text x="137" y="275" text-anchor="middle" dominant-baseline="central" font-size="13" font-weight="500" fill="#0b2026">MCP Client</text>
  <text x="137" y="295" text-anchor="middle" dominant-baseline="central" font-size="11" fill="#618794">Discovers and calls tools</text>

  <rect x="262" y="174" width="386" height="140" rx="12" fill="#2272B4" fill-opacity="0.10"/>
  <rect x="262" y="174" width="386" height="140" rx="12" fill="none" stroke="#2272B4" stroke-width="1.5" stroke-dasharray="6 3"/>
  <text x="638" y="189" text-anchor="end" dominant-baseline="central" font-size="14" font-weight="600" fill="#0b2026">MCP Servers</text>

  <rect x="274" y="204" width="114" height="90" rx="8" fill="#ffffff" stroke="#1B5162" stroke-width="1.2"/>
  <text x="331" y="225" text-anchor="middle" dominant-baseline="central" font-size="13" font-weight="600" fill="#0b2026">Genie</text>
  <text x="331" y="258" text-anchor="middle" dominant-baseline="central" font-size="11" fill="#618794">Structured</text>
  <text x="331" y="274" text-anchor="middle" dominant-baseline="central" font-size="11" fill="#618794">data</text>

  <rect x="398" y="204" width="114" height="90" rx="8" fill="#ffffff" stroke="#1B5162" stroke-width="1.2"/>
  <text x="455" y="225" text-anchor="middle" dominant-baseline="central" font-size="13" font-weight="600" fill="#0b2026">AI Search</text>
  <text x="455" y="258" text-anchor="middle" dominant-baseline="central" font-size="11" fill="#618794">Unstructured</text>
  <text x="455" y="274" text-anchor="middle" dominant-baseline="central" font-size="11" fill="#618794">data</text>

  <rect x="522" y="204" width="114" height="90" rx="8" fill="#ffffff" stroke="#1B5162" stroke-width="1.2"/>
  <text x="579" y="225" text-anchor="middle" dominant-baseline="central" font-size="13" font-weight="600" fill="#0b2026">UC Functions</text>
  <text x="579" y="258" text-anchor="middle" dominant-baseline="central" font-size="11" fill="#618794">Custom</text>
  <text x="579" y="274" text-anchor="middle" dominant-baseline="central" font-size="11" fill="#618794">logic</text>

  <path d="M242 222 L262 222" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#mcp-arrow)"/>
  <path d="M262 270 L242 270" fill="none" stroke="#1B3139" stroke-width="1.8" marker-end="url(#mcp-arrow)"/>
</svg>
</div>

The main benefit of MCP is standardization. You can create a tool once and use it with different agent frameworks or clients that support MCP.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
Read more in the Databricks docs for MCP on Databricks (<a href="https://docs.databricks.com/aws/en/generative-ai/mcp/" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/generative-ai/mcp/" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/generative-ai/mcp/" style="color: #1976d2; text-decoration: underline;">GCP</a>) and managed MCP servers (<a href="https://docs.databricks.com/aws/en/generative-ai/mcp/managed-mcp/" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/generative-ai/mcp/managed-mcp" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/generative-ai/mcp/managed-mcp" style="color: #1976d2; text-decoration: underline;">GCP</a>). For the broader protocol, see the <a href="https://modelcontextprotocol.io/docs/getting-started/intro" style="color: #1976d2; text-decoration: underline;">official MCP documentation</a>.
</p>
</div>
</div>
</div>

<div style="width: 100%; font-family: sans-serif;">

<div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">

  <img src="./Includes/images/lecture 1/genie-code.png" style="height: 64px; margin-bottom: 10px;">

  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">
    Want to know more about MCP on Databricks? Ask Genie Code. Click on the genie icon <img src="./Includes/images/lecture 1/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.
  </div>

  <div style="display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0b2026;">
    <span id="genie-query" style="flex: 1;">I need a review of MCP on Databricks. Can I access MCP tools via the UI?</span>
    <button onclick="
      var text = document.getElementById('genie-query').innerText;
      var ta = document.createElement('textarea');
      ta.value = text;
      ta.style.position = 'fixed';
      ta.style.opacity = '0';
      document.body.appendChild(ta);
      ta.select();
      document.execCommand('copy');
      document.body.removeChild(ta);
      this.innerText = 'Copied!';
      var btn = this;
      setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>

</div>

</div>

### B2. Other Tools
While we will focus on agent tools in UC, it's important to point out that there are other tools that exist outside those discussed in this course. 

#### AI Search
Databricks AI Search (formerly Databricks Vector Search) is a vector search solution that is built into the Databricks Data Intelligence Platform and integrated with its governance and productivity tools. Vector search is a type of search optimized for retrieving embeddings.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
You can read more about AI Search here (<a href="https://docs.databricks.com/aws/en/ai-search/ai-search" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/ai-search/ai-search" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/ai-search/ai-search" style="color: #1976d2; text-decoration: underline;">GCP</a>).
</p>
</div>
</div>
</div>

#### Common Tool Patterns
Below is a summary of some common tool patterns, as well as additional links for reading, that exist on Databricks today. 
| Tool pattern| Description|
|-------------|------------|
| **Structured data retrieval tools ([AWS](https://docs.databricks.com/aws/en/generative-ai/agent-framework/structured-retrieval-tools) \| [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/structured-retrieval-tools) \| [GCP](https://docs.databricks.com/gcp/en/generative-ai/agent-framework/structured-retrieval-tools))** | Query SQL tables, databases, and structured data sources.                                   |
| **Unstructured data retrieval tools ([AWS](https://docs.databricks.com/aws/en/generative-ai/agent-framework/unstructured-retrieval-tools) \| [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/unstructured-retrieval-tools) \| [GCP](https://docs.databricks.com/gcp/en/generative-ai/agent-framework/unstructured-retrieval-tools))** | Search document collections and perform retrieval-augmented generation.                    |
| **Code interpreter tools ([AWS](https://docs.databricks.com/aws/en/generative-ai/agent-framework/code-interpreter-tools) \| [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/code-interpreter-tools) \| [GCP](https://docs.databricks.com/gcp/en/generative-ai/agent-framework/code-interpreter-tools))**         | Allow agents to run Python code for calculations, data analysis, and dynamic processing.   |
| **External connection tools ([AWS](https://docs.databricks.com/aws/en/generative-ai/agent-framework/external-connection-tools) \| [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/external-connection-tools) \| [GCP](https://docs.databricks.com/gcp/en/generative-ai/agent-framework/external-connection-tools))**      | Connect to external services and APIs such as Slack.                                        |
| **AI Playground prototyping ([AWS](https://docs.databricks.com/aws/en/generative-ai/agent-framework/ai-playground-agent) \| [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/ai-playground-agent) \| [GCP](https://docs.databricks.com/gcp/en/generative-ai/agent-framework/ai-playground-agent))**      | Use the AI Playground to quickly add Unity Catalog tools to agents and prototype behavior. |

## C. Governance and AI Management with Unity AI Gateway (Beta)

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 680 450" role="img" style="font-family: sans-serif;">
  <title>Unity AI Gateway Architecture</title>
  <desc>Client sends requests through Unity AI Gateway which provides permission/rate limiting, payload logging, usage tracking, AI guardrails, fallbacks, and traffic splitting. Gateway routes to External Models, Databricks Hosted models, or MCP. Databricks Hosted models offer pay-per-token or provisioned throughput.</desc>
  <defs>
    <marker id="gw-arrow-green" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#3B8B6E" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
    <marker id="gw-arrow-gray" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#8B9DAA" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>

  <!-- Client box (pill shape) -->
  <rect x="258" y="8" width="164" height="52" rx="26" fill="#ffffff" stroke="#8B9DAA" stroke-width="1.5"/>
  <text x="340" y="26" text-anchor="middle" font-size="14" font-weight="600" fill="#0b2026">Client</text>
  <text x="340" y="44" text-anchor="middle" font-size="13" fill="#8B9DAA">(user/app)</text>

  <!-- Arrow: Client to Gateway -->
  <path d="M340 60 L340 86" fill="none" stroke="#3B8B6E" stroke-width="2" marker-end="url(#gw-arrow-green)"/>

  <!-- Gateway container (y=90, h=168, bottom=258) -->
  <rect x="88" y="90" width="504" height="168" rx="14" fill="#EEF4F0" stroke="#3B8B6E" stroke-width="2.5"/>
  <text x="340" y="118" text-anchor="middle" font-size="20" font-weight="700" fill="#3B8B6E">Unity AI Gateway</text>

  <!-- Row 1 feature boxes (y=132, h=46, center=155) -->
  <rect x="108" y="132" width="148" height="46" rx="12" fill="#ffffff" stroke="#D4DAD6" stroke-width="1.2"/>
  <text x="182" y="149" text-anchor="middle" font-size="12.5" fill="#0b2026">Permission and</text>
  <text x="182" y="165" text-anchor="middle" font-size="12.5" fill="#0b2026">rate limiting</text>

  <rect x="268" y="132" width="138" height="46" rx="12" fill="#ffffff" stroke="#D4DAD6" stroke-width="1.2"/>
  <text x="337" y="155" text-anchor="middle" dominant-baseline="central" font-size="12.5" fill="#0b2026">Payload logging</text>

  <rect x="418" y="132" width="148" height="46" rx="12" fill="#ffffff" stroke="#D4DAD6" stroke-width="1.2"/>
  <text x="492" y="155" text-anchor="middle" dominant-baseline="central" font-size="12.5" fill="#0b2026">Usage tracking</text>

  <!-- Row 2 feature boxes (y=190, h=46, center=213) -->
  <rect x="108" y="190" width="138" height="46" rx="12" fill="#ffffff" stroke="#D4DAD6" stroke-width="1.2"/>
  <text x="177" y="213" text-anchor="middle" dominant-baseline="central" font-size="12.5" fill="#0b2026">AI Guardrails</text>

  <rect x="258" y="190" width="138" height="46" rx="12" fill="#ffffff" stroke="#D4DAD6" stroke-width="1.2"/>
  <text x="327" y="213" text-anchor="middle" dominant-baseline="central" font-size="12.5" fill="#0b2026">Fallbacks</text>

  <rect x="408" y="190" width="158" height="46" rx="12" fill="#ffffff" stroke="#D4DAD6" stroke-width="1.2"/>
  <text x="487" y="213" text-anchor="middle" dominant-baseline="central" font-size="12.5" fill="#0b2026">Traffic splitting</text>

  <!-- Arrows: Gateway to three targets -->
  <path d="M210 258 L210 268 L133 268 L133 296" fill="none" stroke="#8B9DAA" stroke-width="1.5" marker-end="url(#gw-arrow-gray)"/>
  <path d="M340 258 L340 296" fill="none" stroke="#8B9DAA" stroke-width="1.5" marker-end="url(#gw-arrow-gray)"/>
  <path d="M470 258 L470 268 L553 268 L553 296" fill="none" stroke="#8B9DAA" stroke-width="1.5" marker-end="url(#gw-arrow-gray)"/>

  <!-- Target boxes (y=300, h=48, center=324) -->
  <rect x="38" y="300" width="190" height="48" rx="8" fill="#ffffff" stroke="#8B9DAA" stroke-width="1.5"/>
  <text x="133" y="324" text-anchor="middle" dominant-baseline="central" font-size="14" fill="#0b2026">External Models</text>

  <rect x="248" y="300" width="190" height="48" rx="8" fill="#ffffff" stroke="#8B9DAA" stroke-width="1.5"/>
  <text x="343" y="318" text-anchor="middle" font-size="14" fill="#0b2026">Databricks Hosted</text>
  <text x="343" y="334" text-anchor="middle" font-size="14" fill="#0b2026">models</text>

  <rect x="458" y="300" width="190" height="48" rx="8" fill="#ffffff" stroke="#8B9DAA" stroke-width="1.5"/>
  <text x="553" y="324" text-anchor="middle" dominant-baseline="central" font-size="14" fill="#0b2026">MCP</text>

  <!-- Arrows: Databricks Hosted to sub-options -->
  <path d="M300 348 L300 358 L268 358 L268 386" fill="none" stroke="#8B9DAA" stroke-width="1.5" marker-end="url(#gw-arrow-gray)"/>
  <path d="M386 348 L386 358 L420 358 L420 386" fill="none" stroke="#8B9DAA" stroke-width="1.5" marker-end="url(#gw-arrow-gray)"/>

  <!-- Sub-option boxes (y=390, h=48, center=414) -->
  <rect x="178" y="390" width="170" height="48" rx="8" fill="#ffffff" stroke="#8B9DAA" stroke-width="1.5"/>
  <text x="263" y="414" text-anchor="middle" dominant-baseline="central" font-size="14" fill="#0b2026">Pay-per-token</text>

  <rect x="368" y="390" width="190" height="48" rx="8" fill="#ffffff" stroke="#8B9DAA" stroke-width="1.5"/>
  <text x="463" y="408" text-anchor="middle" font-size="14" fill="#0b2026">Provisioned</text>
  <text x="463" y="424" text-anchor="middle" font-size="14" fill="#0b2026">throughput</text>
</svg>
</div>

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #4299E0); color: white; padding: 12px 18px; cursor: pointer; font-weight: 600; font-size: 12pt; border-radius: 8px; user-select: none;">
What does Unity AI Gateway provide?
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 16px 20px; background: #F9F7F4; font-size: 12pt; line-height: 1.7; color: #333;">

<p style="margin-top: 10px;"><a href="https://www.databricks.com/blog/ai-gateway-governance-layer-agentic-ai" style="color: #1976d2; text-decoration: underline;">Unity AI Gateway</a> is the Databricks central AI governance layer for LLM endpoints (<a href="https://docs.databricks.com/aws/en/ai-gateway/#llms" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/ai-gateway/#llms" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/ai-gateway/#llms" style="color: #1976d2; text-decoration: underline;">GCP</a>), MCP servers (<a href="https://docs.databricks.com/aws/en/ai-gateway/#mcps" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/ai-gateway/#mcps" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/ai-gateway/#mcps" style="color: #1976d2; text-decoration: underline;">GCP</a>), and coding agents. Use Unity AI Gateway to analyze usage, configure permissions, enforce guardrails, and manage capacity across providers.</p>

<ul style="margin: 12px 0; padding-left: 24px;">
<li><strong>Governance</strong>: Track which users, agents, and applications are making LLM calls, how agents use MCP servers and APIs, and applies consistent policies across models and tools
<ul style="margin: 4px 0; padding-left: 20px;"><li><strong>MCP Governance</strong>: On-behalf-of execution so agents act with the request user's exact permissions instead of a shared service account.</li></ul></li>
<li><strong>Rate limiting</strong>: Protect endpoints from overload with per-endpoint and per-identity (user or group) QPM/TPM limits (<a href="https://docs.databricks.com/aws/en/ai-gateway/rate-limits-beta" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/ai-gateway/rate-limits-beta" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/ai-gateway/rate-limits-beta" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
<li><strong>Guardrails</strong>: Apply input/output safety filters and data-protection policies to block or mask harmful or sensitive content</li>
<li><strong>Usage tracking</strong>: Monitor token consumption and costs across Unity AI Gateway endpoints and coding agents using system tables</li>
<li><strong>Payload logging</strong>: Log request/response payloads to Unity Catalog Delta tables via inference tables for audit, debugging, and model improvement</li>
</ul>

</div>
</details>

### C1. Fallback Example
<p>Without Unity AI Gateway, each agent or application would have to implement its own access control, guardrails, and logging, making safety, compliance, debugging, and cost control much harder to do reliably at scale (especially across multiple models and providers). The diagram below illustrates how Unity AI Gateway handles endpoint fallback routing.</p>

</div>
</details>

<div style="max-width: 900px; margin: 0 auto;">
<svg width="100%" viewBox="0 0 680 290" role="img" style="font-family: sans-serif;">
  <title>Unity AI Gateway Endpoint fallback routing</title>
  <desc>Requests route from a client to Model 1, falling back to Model 2 or Model 3 on 429/5xx errors, with all outcomes logged to inference and usage tracking tables.</desc>
  <defs>
    <marker id="fb-green" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#00A972" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
    <marker id="fb-red" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#98102A" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
    <marker id="fb-gray" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#618794" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>

  <!-- Gateway container -->
  <rect x="120" y="20" width="540" height="258" rx="16" fill="#F9F7F4" stroke="#1B3139" stroke-width="1.5" stroke-dasharray="7 4" stroke-opacity="0.4"/>
  <text x="390" y="46" text-anchor="middle" font-size="13" fill="#618794">Unity AI Gateway Endpoint</text>

  <!-- Log box -->
  <rect x="275" y="60" width="230" height="56" rx="8" fill="#EEEDE9" stroke="#00A972" stroke-width="1.5"/>
  <text x="390" y="84" text-anchor="middle" font-size="13" font-weight="600" fill="#0b2026">Request logged</text>
  <text x="390" y="102" text-anchor="middle" font-size="11" fill="#618794">Inference &amp; usage tracking table</text>

  <!-- Routing group -->
  <rect x="135" y="146" width="520" height="102" rx="12" fill="#2272B4" fill-opacity="0.08" stroke="#2272B4" stroke-width="1.5" stroke-dasharray="6 3"/>
  <text x="649" y="162" text-anchor="end" font-size="11" font-style="italic" fill="#2272B4">Routing targets</text>

  <!-- Model 1 -->
  <rect x="148" y="158" width="132" height="78" rx="8" fill="#ffffff" stroke="#1B5162" stroke-width="1.5"/>
  <text x="214" y="184" text-anchor="middle" font-size="14" font-weight="600" fill="#0b2026">Model 1</text>
  <rect x="155" y="193" width="118" height="18" rx="4" fill="#FFAB00"/>
  <text x="214" y="202" text-anchor="middle" font-size="10" font-weight="600" fill="#0b2026" dominant-baseline="central">Original request</text>
  <text x="214" y="224" text-anchor="middle" font-size="11" fill="#618794">Primary route</text>

  <!-- Model 2 -->
  <rect x="340" y="178" width="130" height="56" rx="8" fill="#ffffff" stroke="#1B5162" stroke-width="1.5"/>
  <text x="405" y="202" text-anchor="middle" font-size="14" font-weight="600" fill="#0b2026">Model 2</text>
  <text x="405" y="221" text-anchor="middle" font-size="11" fill="#618794">First fallback</text>

  <!-- Model 3 -->
  <rect x="530" y="178" width="110" height="56" rx="8" fill="#ffffff" stroke="#1B5162" stroke-width="1.5"/>
  <text x="585" y="202" text-anchor="middle" font-size="14" font-weight="600" fill="#0b2026">Model 3</text>
  <text x="585" y="221" text-anchor="middle" font-size="11" fill="#618794">Last fallback</text>

  <!-- Green: Model 1 to Log -->
  <path d="M214 158 L310 116" fill="none" stroke="#00A972" stroke-width="1.8" marker-end="url(#fb-green)"/>
  <text x="218" y="131" text-anchor="end" font-size="11" fill="#00A972">200 (success)</text>

  <!-- Red: Model 1 to Model 2 -->
  <path d="M280 205 L340 205" fill="none" stroke="#98102A" stroke-width="1.8" marker-end="url(#fb-red)"/>
  <text x="310" y="198" text-anchor="middle" font-size="11" fill="#98102A">429/5xx</text>

  <!-- Green: Model 2 to Log -->
  <path d="M405 178 L390 116" fill="none" stroke="#00A972" stroke-width="1.8" marker-end="url(#fb-green)"/>
  <text x="406" y="131" text-anchor="start" font-size="11" fill="#00A972">200</text>

  <!-- Red: Model 2 to Model 3 -->
  <path d="M470 205 L530 205" fill="none" stroke="#98102A" stroke-width="1.8" stroke-dasharray="6 3" marker-end="url(#fb-red)"/>
  <text x="500" y="198" text-anchor="middle" font-size="11" fill="#98102A">429/5xx</text>

  <!-- Green: Model 3 to Log -->
  <path d="M585 178 L470 116" fill="none" stroke="#00A972" stroke-width="1.8" marker-end="url(#fb-green)"/>
  <text x="505" y="131" text-anchor="start" font-size="11" fill="#00A972">200</text>

  <!-- Client -->
  <rect x="28" y="186" width="64" height="38" rx="8" fill="#ffffff" stroke="#618794" stroke-width="1.5"/>
  <text x="60" y="205" text-anchor="middle" font-size="13" fill="#0b2026" dominant-baseline="central">Client</text>
  <path d="M92 205 L148 205" fill="none" stroke="#618794" stroke-width="1.8" marker-end="url(#fb-gray)"/>
</svg>
</div>


<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
Databricks Foundation Model APIs integrate with Unity AI Gateway, but you enable Unity AI Gateway features (like usage tracking and inference tables) per endpoint. In the new Unity AI Gateway (Beta), usage tracking is on by default for Gateway endpoints, once the feature is enabled and the endpoint is created.
See: Unity AI Gateway documentation (<a href="https://docs.databricks.com/aws/en/ai-gateway/overview-beta" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/ai-gateway/overview-beta" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/ai-gateway/overview-beta" style="color: #1976d2; text-decoration: underline;">GCP</a>)
</p>
</div>
</div>
</div>


## D. Conclusion

In this lecture, you learned that AI agents are autonomous systems that combine LLM reasoning with tool execution. On Databricks, agent tools are Unity Catalog functions -- governed, discoverable, and secure. MCP provides the standard protocol for agents to discover and call these tools at runtime, while AI Gateway adds a governance layer across LLM endpoints and MCP servers.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>